# VSX–TESS Data Pipeline: What, Why, and How

This notebook documents the data pipeline used to construct a labeled variable-star light-curve dataset. The project starts with variable-star categories from **AAVSO VSX**, maps VSX sources to likely **TESS Input Catalog (TIC)** identifiers, and retrieves usable TESS light curves through a hierarchy of **SPOC → QLP → TESSCut**.

The notebook is intended to serve as both a demonstration and a methodology document.

## 1. Why choose AAVSO VSX as the label source?

AAVSO VSX is used because it provides a large public catalog of known variable stars with sky coordinates and variability classifications. For this research project, VSX supplies the supervised-learning labels: the variable-star class or family.

TESS provides high-quality photometry, but it does not by itself provide a clean labeled dataset for all variable-star classes. VSX fills that role. The tradeoff is that VSX is not already linked one-to-one to TIC identifiers, so the pipeline must solve a crossmatching problem before light curves can be downloaded.

## 2. Why choose TESS light curves as the data source?

TESS is chosen because it provides large-scale, space-based time-domain photometry across much of the sky. Variable stars are naturally studied through light curves, and TESS data are well suited for period analysis, feature extraction, and machine-learning classification.

TESS data can be accessed through several product types:

- **SPOC**: high-quality pipeline light curves, but limited coverage.
- **QLP**: broader availability, useful when SPOC is missing.
- **TESSCut**: fallback access to full-frame image cutouts, enabling custom light-curve extraction.

The key lesson from this project is that SPOC/QLP alone miss most usable VSX targets; TESSCut is essential for high coverage.

## 3. Imports and representative examples

We use three stars to demonstrate the logic:

| VSX ID | RA (deg) | Dec (deg) | Final recovery path |
|---|---:|---:|---|
| LMC V0827 | 76.02176 | -66.29695 | TESSCut |
| OGLE-GD-CEP-0948 | 206.05846 | -63.67383 | SPOC |
| ZTF J212148.91+495425.9 | 320.45383 | 49.90721 | QLP |

In [ ]:
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.io import fits

try:
    from astroquery.mast import Catalogs
    ASTROQUERY_AVAILABLE = True
except Exception as e:
    ASTROQUERY_AVAILABLE = False
    print("astroquery is not available:", e)

try:
    import lightkurve as lk
    LIGHTKURVE_AVAILABLE = True
except Exception as e:
    LIGHTKURVE_AVAILABLE = False
    print("lightkurve is not available:", e)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

In [ ]:
vsx_examples = pd.DataFrame([
    {"vsx_id": "LMC V0827", "ra_deg": 76.02176, "dec_deg": -66.29695, "expected_final_method": "TESSCut"},
    {"vsx_id": "OGLE-GD-CEP-0948", "ra_deg": 206.05846, "dec_deg": -63.67383, "expected_final_method": "SPOC"},
    {"vsx_id": "ZTF J212148.91+495425.9", "ra_deg": 320.45383, "dec_deg": 49.90721, "expected_final_method": "QLP"},
])
vsx_examples

## 4. Original idea: nearest TIC by RA/Dec

The original idea was:

1. take the VSX RA/Dec,
2. query nearby TIC sources,
3. compute angular separation,
4. pick the nearest TIC source,
5. search for a SPOC light curve.

This is a reasonable baseline, but it is incomplete. A correct TIC counterpart may lack a SPOC product, and in crowded fields a single nearest match can be fragile.

In [ ]:
def query_tic_candidates(ra_deg, dec_deg, radius_arcsec=10.0, max_candidates=5):
    if not ASTROQUERY_AVAILABLE:
        raise RuntimeError("astroquery is not available in this environment.")

    vsx_coord = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame="icrs")
    tic_table = Catalogs.query_region(vsx_coord, radius=radius_arcsec * u.arcsec, catalog="TIC")

    rows = []
    for row in tic_table:
        if "ra" not in row.colnames or "dec" not in row.colnames or "ID" not in row.colnames:
            continue

        tic_ra = float(row["ra"])
        tic_dec = float(row["dec"])
        tic_coord = SkyCoord(ra=tic_ra * u.deg, dec=tic_dec * u.deg, frame="icrs")
        sep_arcsec = vsx_coord.separation(tic_coord).arcsec

        tmag = np.nan
        if "Tmag" in row.colnames:
            try:
                tmag = float(row["Tmag"])
            except Exception:
                pass

        rows.append({
            "tic_id": str(row["ID"]),
            "tic_ra_deg": tic_ra,
            "tic_dec_deg": tic_dec,
            "tic_tmag": tmag,
            "sep_arcsec": float(sep_arcsec),
        })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df
    return df.sort_values("sep_arcsec").head(max_candidates).reset_index(drop=True)

In [ ]:
candidate_tables = {}
for _, star in vsx_examples.iterrows():
    try:
        cand = query_tic_candidates(star["ra_deg"], star["dec_deg"], radius_arcsec=10.0, max_candidates=5)
        candidate_tables[star["vsx_id"]] = cand
        print("\n" + "=" * 80)
        print(star["vsx_id"])
        display(cand)
    except Exception as e:
        print(f"Could not query TIC candidates for {star['vsx_id']}: {e}")

## 5. SPOC-only search: first bottleneck

Using the nearest TIC and searching only SPOC recovers only the SPOC example. This demonstrates that **missing SPOC does not mean missing TESS data**.

In [ ]:
known_examples = pd.DataFrame([
    {"vsx_id": "LMC V0827", "tic_id": "30855441", "expected_final_method": "TESSCut"},
    {"vsx_id": "OGLE-GD-CEP-0948", "tic_id": "321957546", "expected_final_method": "SPOC"},
    {"vsx_id": "ZTF J212148.91+495425.9", "tic_id": "63711350", "expected_final_method": "QLP"},
])

def count_lightcurve_products(tic_id, author):
    if not LIGHTKURVE_AVAILABLE:
        return np.nan
    try:
        return len(lk.search_lightcurve(f"TIC {tic_id}", mission="TESS", author=author))
    except Exception as e:
        print(f"Warning: search_lightcurve failed for TIC {tic_id}, author={author}: {e}")
        return np.nan

def count_tesscut_products(tic_id):
    if not LIGHTKURVE_AVAILABLE:
        return np.nan
    try:
        return len(lk.search_tesscut(f"TIC {tic_id}"))
    except Exception as e:
        print(f"Warning: search_tesscut failed for TIC {tic_id}: {e}")
        return np.nan

spoc_demo = known_examples.copy()
spoc_demo["SPOC_count"] = spoc_demo["tic_id"].apply(lambda x: count_lightcurve_products(x, "SPOC"))
spoc_demo["SPOC_found"] = spoc_demo["SPOC_count"] > 0
spoc_demo

## 6. Improvement: top-5 candidates and SPOC → QLP

The improved pipeline stores up to five TIC candidates per VSX star. It then searches each candidate in distance order, first for SPOC and then for QLP.

This recovers cases like **ZTF J212148.91+495425.9**, which lacks SPOC but has QLP.

In [ ]:
spoc_qlp_demo = known_examples.copy()
spoc_qlp_demo["SPOC_count"] = spoc_qlp_demo["tic_id"].apply(lambda x: count_lightcurve_products(x, "SPOC"))
spoc_qlp_demo["QLP_count"] = spoc_qlp_demo["tic_id"].apply(lambda x: count_lightcurve_products(x, "QLP"))

def select_spoc_or_qlp(row):
    if row["SPOC_count"] > 0:
        return "SPOC"
    if row["QLP_count"] > 0:
        return "QLP"
    return None

spoc_qlp_demo["Selected_after_SPOC_QLP"] = spoc_qlp_demo.apply(select_spoc_or_qlp, axis=1)
spoc_qlp_demo

## 7. Final fallback: TESSCut

If no candidate has SPOC or QLP, the pipeline uses TESSCut at the nearest TIC candidate coordinate. TESSCut provides full-frame image cutouts from which the pipeline extracts a 1D light curve.

This recovers cases like **LMC V0827**.

In [ ]:
final_demo = spoc_qlp_demo.copy()
final_demo["TESSCut_count"] = final_demo["tic_id"].apply(count_tesscut_products)

def select_final_method(row):
    if row["SPOC_count"] > 0:
        return "SPOC"
    if row["QLP_count"] > 0:
        return "QLP"
    if row["TESSCut_count"] > 0:
        return "TESSCut"
    return "Missing"

final_demo["Final_selected_method"] = final_demo.apply(select_final_method, axis=1)
final_demo

## 8. Final pipeline logic

```text
For each VSX star:
    query TIC candidates near VSX RA/Dec
    keep up to 5 candidates sorted by angular distance

Metadata phase:
    batch query unique TIC candidates for SPOC/QLP availability
    select first candidate with SPOC, otherwise first with QLP
    if none, mark bestMatch = None

Download phase:
    if bestMatch exists: download SPOC/QLP
    otherwise: use nearest TIC candidate for TESSCut extraction

Storage:
    save raw FITS
    save standardized FITS
    save metadata and QC status
```

## 9. Raw and standardized light curves

A key issue solved during development was avoiding blind standardization. Standardizing flux too early can erase information needed for physical interpretation, such as amplitude or original flux scale.

The final pipeline saves:

1. **raw light curve**: native flux values,
2. **standardized light curve**: cleaned and normalized version for ML.

Standardization is skipped or flagged when the light curve is all-NaN, has too few valid points, or has near-zero variance.

## 10. FITS sanity-check logic

The pipeline includes a FITS validation script based on the following checks:

- can the file be opened?
- is there a recognizable TIME/FLUX table?
- does it look like a final 1D light curve rather than an image cube?
- how many finite time/flux rows are present?
- is the finite fraction acceptable?
- is the time span positive?
- is the flux scatter nonzero?
- is the file suspiciously small?

This protects the downstream feature-extraction and ML stages.

In [ ]:
TIME_COL_CANDIDATES = ["TIME", "BJD", "TIMECORR"]
FLUX_COL_CANDIDATES = ["FLUX", "SAP_FLUX", "PDCSAP_FLUX", "KSPSAP_FLUX", "RAW_FLUX"]

@dataclass
class FitsScanResult:
    path: str
    size_bytes: int
    verdict: str
    suspicious: bool
    has_time_flux_table: bool
    has_image_hdu: bool
    hdu_count: int
    table_hdu_index: Optional[int]
    time_col: Optional[str]
    flux_col: Optional[str]
    total_rows: Optional[int]
    finite_rows: Optional[int]
    finite_fraction: Optional[float]
    time_span: Optional[float]
    flux_std: Optional[float]
    issues: str

def find_time_flux_pair(hdul):
    for i, hdu in enumerate(hdul):
        data = hdu.data
        names = getattr(data, "names", None)
        if names is None:
            continue
        names_upper = {n.upper(): n for n in names}
        time_col = next((names_upper[t] for t in TIME_COL_CANDIDATES if t in names_upper), None)
        flux_col = next((names_upper[f] for f in FLUX_COL_CANDIDATES if f in names_upper), None)
        if time_col and flux_col:
            return i, time_col, flux_col
    return None, None, None

def classify_fits_file(path, min_size_kb=50, min_finite_rows=100, min_finite_fraction=0.50):
    path = Path(path)
    issues = []
    size_bytes = path.stat().st_size

    if size_bytes < min_size_kb * 1024:
        issues.append(f"small_file<{min_size_kb}KB")

    try:
        with fits.open(path) as hdul:
            hdu_count = len(hdul)
            has_image = any(isinstance(h.data, np.ndarray) and h.data.ndim >= 2 and h.data.size > 0 for h in hdul)
            hdu_idx, time_col, flux_col = find_time_flux_pair(hdul)
            has_time_flux = hdu_idx is not None

            total_rows = finite_rows = finite_fraction = time_span = flux_std = None

            if has_time_flux:
                table = hdul[hdu_idx].data
                total_rows = len(table)
                time = np.asarray(table[time_col], dtype=float)
                flux = np.asarray(table[flux_col], dtype=float)
                finite = np.isfinite(time) & np.isfinite(flux)
                finite_rows = int(finite.sum())
                finite_fraction = finite_rows / total_rows if total_rows else None
                if finite_rows > 0:
                    time_span = float(np.nanmax(time[finite]) - np.nanmin(time[finite]))
                    flux_std = float(np.nanstd(flux[finite]))

            if has_time_flux:
                verdict = "LIKELY_FINAL_LIGHT_CURVE_FITS"
            elif has_image:
                verdict = "LIKELY_IMAGE_OR_TESSCUT_PRODUCT"
            else:
                verdict = "UNCLEAR_OR_METADATA_ONLY"

            if not has_time_flux:
                issues.append("no_recognizable_time_flux_table")
            if total_rows == 0:
                issues.append("empty_time_flux_table")
            if finite_rows is not None and finite_rows < min_finite_rows:
                issues.append(f"too_few_finite_rows<{min_finite_rows}")
            if finite_fraction is not None and finite_fraction < min_finite_fraction:
                issues.append(f"low_finite_fraction<{min_finite_fraction:.2f}")
            if time_span is not None and time_span <= 0:
                issues.append("non_positive_time_span")
            if flux_std is not None and flux_std == 0:
                issues.append("zero_flux_scatter")

            return FitsScanResult(
                path=str(path), size_bytes=size_bytes, verdict=verdict,
                suspicious=len(issues) > 0, has_time_flux_table=has_time_flux,
                has_image_hdu=has_image, hdu_count=hdu_count, table_hdu_index=hdu_idx,
                time_col=time_col, flux_col=flux_col, total_rows=total_rows,
                finite_rows=finite_rows, finite_fraction=finite_fraction,
                time_span=time_span, flux_std=flux_std, issues=";".join(issues)
            )

    except Exception as e:
        return FitsScanResult(
            path=str(path), size_bytes=size_bytes, verdict="FAILED_TO_OPEN",
            suspicious=True, has_time_flux_table=False, has_image_hdu=False,
            hdu_count=0, table_hdu_index=None, time_col=None, flux_col=None,
            total_rows=None, finite_rows=None, finite_fraction=None,
            time_span=None, flux_std=None, issues=f"open_error:{type(e).__name__}:{e}"
        )

In [ ]:
def determine_qc_status(scan_result):
    status = "pass"
    issues = []

    if scan_result.verdict == "FAILED_TO_OPEN":
        status = "fail"
        issues.append("failed_to_open")

    if scan_result.verdict == "UNCLEAR_OR_METADATA_ONLY" and not scan_result.has_time_flux_table:
        status = "fail"
        issues.append("no_time_flux_table")

    if scan_result.finite_rows is not None and scan_result.finite_rows < 100:
        status = "fail"
        issues.append("too_few_finite_rows")

    if scan_result.finite_fraction is not None and scan_result.finite_fraction < 0.50:
        status = "fail"
        issues.append("low_finite_fraction")

    if status != "fail":
        warning_flags = []
        if scan_result.finite_fraction is not None and 0.50 <= scan_result.finite_fraction < 0.75:
            warning_flags.append("marginal_finite_fraction")
        if scan_result.finite_rows is not None and 100 <= scan_result.finite_rows < 300:
            warning_flags.append("marginal_row_count")
        if scan_result.size_bytes < 50 * 1024:
            warning_flags.append("small_file")
        if warning_flags:
            status = "warning"
            issues.extend(warning_flags)

    return status, ";".join(issues) if issues else "all_checks_passed"

## 11. Practical issues solved during pipeline development

The following issues were identified and addressed:

- **Non-unique VSX → TIC matching:** solved by keeping up to five TIC candidates.
- **Very low SPOC-only coverage:** initial coverage was approximately 12%, with RR Lyrae initially around 1.4%.
- **Efficient availability lookup:** solved by batch-querying TIC candidates for SPOC/QLP availability.
- **TESSCut fallback:** added to recover stars missing from SPOC/QLP.
- **Candidate-centered TESSCut:** fallback uses nearest TIC candidate coordinates.
- **Raw vs standardized preservation:** both versions are saved.
- **FITS sanity checking:** added to detect bad files before feature extraction.
- **Conservative parallelism:** large download was run with limited concurrency for stability.
- **Long-trend handling:** deferred to a later preprocessing module so that real astrophysical variability is not removed blindly.

## 12. Final coverage results

The final pipeline was applied to **7,370 variable-star candidates**.

| Provenance | Count | Percent |
|---|---:|---:|
| TESSCut | 5,034 | 68.30% |
| SPOC | 467 | 6.34% |
| QLP | 1,612 | 21.87% |
| Missing | 257 | 3.49% |

Final usable coverage is approximately **96.5%**.

Earlier stages:

- nearest TIC + SPOC only: approximately **12%**
- top-5 candidates + SPOC/QLP: approximately **22.5%**
- final SPOC/QLP/TESSCut pipeline: **96.5%**

In [ ]:
overall_provenance = pd.DataFrame([
    {"Provenance": "TESSCut", "Count": 5034, "Percent": 68.30},
    {"Provenance": "SPOC", "Count": 467, "Percent": 6.34},
    {"Provenance": "QLP", "Count": 1612, "Percent": 21.87},
    {"Provenance": "Missing", "Count": 257, "Percent": 3.49},
])

coverage_stages = pd.DataFrame([
    {"Stage": "Nearest TIC + SPOC only\n(initial idea)", "CoveragePercent": 12.0},
    {"Stage": "Top-5 candidates +\nSPOC/QLP", "CoveragePercent": 22.5},
    {"Stage": "Top-5 candidates +\nSPOC/QLP/TESSCut", "CoveragePercent": 96.51},
])

display(overall_provenance)
display(coverage_stages)

### Figure 1. Coverage improvement across pipeline stages

**Caption:** Usable TESS light curve coverage improves from approximately 12% under the initial nearest-TIC/SPOC-only strategy to 22.5% after allowing up to five TIC candidates and searching SPOC/QLP products. Adding TESSCut fallback increases final usable coverage to 96.5%.

In [ ]:
plt.figure(figsize=(8, 5))
bars = plt.bar(coverage_stages["Stage"], coverage_stages["CoveragePercent"])
plt.ylabel("Usable light curve coverage (%)")
plt.ylim(0, 105)
plt.title("VSX–TESS light curve recovery improves with fallback strategy")

for bar, value in zip(bars, coverage_stages["CoveragePercent"]):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 2, f"{value:.1f}%", ha="center")

plt.tight_layout()
plt.savefig("coverage_improvement_stages.png", dpi=200, bbox_inches="tight")
plt.show()

## 13. Family-level final results

In [ ]:
family_stats = pd.DataFrame([
    {"Family": "CEPHEID", "TESSCut": 639, "SPOC": 29, "QLP": 168, "Missing": 0, "Total": 836},
    {"Family": "CV", "TESSCut": 525, "SPOC": 66, "QLP": 4, "Missing": 0, "Total": 595},
    {"Family": "DSCT_SXPHE", "TESSCut": 766, "SPOC": 34, "QLP": 153, "Missing": 0, "Total": 953},
    {"Family": "ECLIPSING", "TESSCut": 814, "SPOC": 8, "QLP": 174, "Missing": 0, "Total": 996},
    {"Family": "ELLIPSOIDAL_ROT", "TESSCut": 705, "SPOC": 122, "QLP": 170, "Missing": 0, "Total": 997},
    {"Family": "LONG_PERIOD", "TESSCut": 173, "SPOC": 90, "QLP": 717, "Missing": 0, "Total": 980},
    {"Family": "RRLYR", "TESSCut": 934, "SPOC": 3, "QLP": 39, "Missing": 0, "Total": 976},
    {"Family": "XRAY", "TESSCut": 20, "SPOC": 24, "QLP": 12, "Missing": 0, "Total": 56},
    {"Family": "YSO", "TESSCut": 458, "SPOC": 91, "QLP": 175, "Missing": 0, "Total": 724},
])

for col in ["TESSCut", "SPOC", "QLP", "Missing"]:
    family_stats[col + "_pct"] = 100 * family_stats[col] / family_stats["Total"]

family_stats

### Figure 2. Provenance composition by family

**Caption:** Final light curve provenance differs strongly by variable-star family. RR Lyrae, eclipsing, DSCT/SX Phe, and Cepheid stars rely heavily on TESSCut, while long-period variables are primarily recovered through QLP.

In [ ]:
plot_df = family_stats.set_index("Family")[["SPOC_pct", "QLP_pct", "TESSCut_pct", "Missing_pct"]]

plt.figure(figsize=(11, 6))
bottom = np.zeros(len(plot_df))

for col in plot_df.columns:
    values = plot_df[col].values
    plt.bar(plot_df.index, values, bottom=bottom, label=col.replace("_pct", ""))
    bottom += values

plt.ylabel("Fraction of family (%)")
plt.title("Final light curve provenance by variable-star family")
plt.xticks(rotation=35, ha="right")
plt.legend(title="Provenance")
plt.tight_layout()
plt.savefig("family_provenance_breakdown.png", dpi=200, bbox_inches="tight")
plt.show()

## 14. Main findings

1. VSX is a practical label source, but it requires careful crossmatching to TIC.
2. TESS is an excellent light-curve source, but standard products are incomplete.
3. Nearest-TIC + SPOC-only is too restrictive.
4. Top-5 candidate matching plus QLP improves coverage.
5. TESSCut is essential, contributing 5,034 light curves.
6. Final usable coverage reaches 96.5%.
7. Quality control and raw-data preservation are necessary for later feature extraction and ML.

The data pipeline is therefore a major methodological contribution, not merely a download step.

## 15. Future preprocessing: long-trend handling

Long-term trend removal was intentionally deferred. Some trends may be instrumental, but some may be astrophysical, especially for long-period variables. The planned approach is to add a separate trend-detection and detrending module after the raw and standardized datasets are safely preserved.